In [ ]:
# Imports & Setup
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import copy
import warnings
warnings.filterwarnings('ignore')

#  MLP architecture (same as M2/M3)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden=[128, 64, 32], num_classes=5, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# Load data
train_df = pd.read_csv('../data/ocpp_app_layer/Combined/Train.csv')
test_df  = pd.read_csv('../data/ocpp_app_layer/Combined/Test.csv')

label_col    = 'label'
feature_cols = [c for c in train_df.columns
                if c != label_col and
                train_df[c].dtype in ['float64','int64','float32','int32']]
print(f"Features: {len(feature_cols)}")

le = LabelEncoder()
y_train = le.fit_transform(train_df[label_col])
y_test  = le.transform(test_df[label_col])

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values)
X_test  = scaler.transform(test_df[feature_cols].values)

# Train baseline model
def train_model(X_tr, y_tr, input_dim, epochs=50, patience=10,
                jitter_std=0.05, seed=42):
    torch.manual_seed(seed)
    m = MLP(input_dim=input_dim)
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.long)
    best_loss, wait, best_state = float('inf'), 0, None
    m.train()
    for epoch in range(epochs):
        idx = torch.randperm(len(X_t))
        epoch_loss = 0
        for i in range(0, len(X_t), 32):
            xb = X_t[idx[i:i+32]]
            yb = y_t[idx[i:i+32]]
            xb = xb + torch.randn_like(xb) * jitter_std
            optimizer.zero_grad()
            loss = criterion(m(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if epoch_loss < best_loss:
            best_loss, wait = epoch_loss, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    m.load_state_dict(best_state)
    m.eval()
    return m

def evaluate(m, X_np, y_np):
    with torch.no_grad():
        logits = m(torch.tensor(X_np, dtype=torch.float32))
        probs  = torch.softmax(logits, dim=1).numpy()
        preds  = np.argmax(probs, axis=1)
    acc = accuracy_score(y_np, preds)
    f1  = f1_score(y_np, preds, average='macro')
    return acc, f1, preds, probs

print("Training baseline model...")
model_v1 = train_model(X_train, y_train, input_dim=X_train.shape[1])
acc, f1, _, _ = evaluate(model_v1, X_test, y_test)
print(f"Baseline — Acc={acc:.4f}  F1={f1:.4f}")
print(f"Classes: {le.classes_}")

Features: 51
Training baseline model...
Baseline — Acc=0.9992  F1=0.9992
Classes: ['cyberattack_ocpp16_doc_idtag'
 'cyberattack_ocpp16_dos_flooding_heartbeat'
 'cyberattack_ocpp16_fdi_chargingprofile'
 'cyberattack_ocpp16_unauthorized_access' 'normal']


In [2]:
# Experience Replay Buffer

class ReplayBuffer:
    """
    Fixed-size experience replay buffer using reservoir sampling.
    Reservoir sampling ensures each sample has equal probability
    of being retained regardless of arrival order.
    """
    def __init__(self, max_size=500, n_classes=5, seed=42):
        self.max_size    = max_size
        self.n_classes   = n_classes
        self.per_class   = max_size // n_classes
        self.X_buf       = {}
        self.y_buf       = {}
        self.rng         = np.random.RandomState(seed)

    def add(self, X_new, y_new):
        """Add new samples using stratified reservoir sampling."""
        for cls in range(self.n_classes):
            mask  = (y_new == cls)
            X_cls = X_new[mask]
            y_cls = y_new[mask]
            if len(X_cls) == 0:
                continue
            if cls not in self.X_buf:
                # First time seeing this class
                self.X_buf[cls] = X_cls[:self.per_class]
                self.y_buf[cls] = y_cls[:self.per_class]
            else:
                # Reservoir sampling: replace existing samples randomly
                combined_X = np.vstack([self.X_buf[cls], X_cls])
                combined_y = np.concatenate([self.y_buf[cls], y_cls])
                if len(combined_X) > self.per_class:
                    idx = self.rng.choice(len(combined_X),
                                          self.per_class, replace=False)
                    combined_X = combined_X[idx]
                    combined_y = combined_y[idx]
                self.X_buf[cls] = combined_X
                self.y_buf[cls] = combined_y

    def sample(self):
        """Return all buffered samples as arrays."""
        if not self.X_buf:
            return None, None
        X_all = np.vstack(list(self.X_buf.values()))
        y_all = np.concatenate(list(self.y_buf.values()))
        return X_all, y_all

    def size(self):
        if not self.X_buf:
            return 0
        return sum(len(v) for v in self.X_buf.values())

    def class_counts(self):
        return {le.classes_[k]: len(v) for k, v in self.X_buf.items()}

# Initialise buffer with training data
replay_buffer = ReplayBuffer(max_size=500, n_classes=5)
replay_buffer.add(X_train, y_train)

print("Replay Buffer initialised:")
print(f"  Total samples: {replay_buffer.size()}")
print(f"  Class distribution: {replay_buffer.class_counts()}")

Replay Buffer initialised:
  Total samples: 500
  Class distribution: {'cyberattack_ocpp16_doc_idtag': 100, 'cyberattack_ocpp16_dos_flooding_heartbeat': 100, 'cyberattack_ocpp16_fdi_chargingprofile': 100, 'cyberattack_ocpp16_unauthorized_access': 100, 'normal': 100}


In [3]:
# Simulate 5 Drift Episodes
np.random.seed(42)

feat_std = X_train.std(axis=0)
feat_std = np.where(feat_std == 0, 1e-6, feat_std)

def make_drift_episode(episode, n=300):
    """
    Simulate a realistic drift episode.
    Episode 1: slight covariate shift
    Episode 2: moderate shift + dos prior increase
    Episode 3: severe shift + new firmware (fdi dominant)
    Episode 4: recovery (shift reduces)
    Episode 5: new attack pattern (unauthorized dominant)
    """
    configs = {
        1: {'shift': 0.05, 'priors': [0.20, 0.20, 0.20, 0.20, 0.20]},
        2: {'shift': 0.10, 'priors': [0.10, 0.40, 0.15, 0.15, 0.20]},
        3: {'shift': 0.20, 'priors': [0.10, 0.20, 0.40, 0.15, 0.15]},
        4: {'shift': 0.08, 'priors': [0.20, 0.25, 0.20, 0.20, 0.15]},
        5: {'shift': 0.15, 'priors': [0.10, 0.15, 0.15, 0.40, 0.20]},
    }
    cfg     = configs[episode]
    priors  = cfg['priors']
    shift   = cfg['shift']

    # Sample according to class priors
    samples_per_class = [int(p * n) for p in priors]
    X_ep, y_ep = [], []
    for cls in range(5):
        cls_idx = np.where(y_train == cls)[0]
        chosen  = np.random.choice(cls_idx,
                                   samples_per_class[cls],
                                   replace=True)
        X_cls = X_train[chosen] + np.random.normal(
            0, shift * feat_std, (len(chosen), X_train.shape[1]))
        X_ep.append(X_cls)
        y_ep.append(np.full(len(chosen), cls))

    X_ep = np.vstack(X_ep)
    y_ep = np.concatenate(y_ep)
    shuffle = np.random.permutation(len(X_ep))
    return X_ep[shuffle], y_ep[shuffle]

# Generate all episodes
episodes = {}
for ep in range(1, 6):
    X_ep, y_ep = make_drift_episode(ep)
    episodes[ep] = {'X': X_ep, 'y': y_ep}
    unique, counts = np.unique(y_ep, return_counts=True)
    print(f"Episode {ep} (shift={[0.05,0.10,0.20,0.08,0.15][ep-1]:.2f}): "
          f"{len(X_ep)} samples — "
          f"{dict(zip(le.classes_[unique], counts))}")

Episode 1 (shift=0.05): 300 samples — {'cyberattack_ocpp16_doc_idtag': np.int64(60), 'cyberattack_ocpp16_dos_flooding_heartbeat': np.int64(60), 'cyberattack_ocpp16_fdi_chargingprofile': np.int64(60), 'cyberattack_ocpp16_unauthorized_access': np.int64(60), 'normal': np.int64(60)}
Episode 2 (shift=0.10): 300 samples — {'cyberattack_ocpp16_doc_idtag': np.int64(30), 'cyberattack_ocpp16_dos_flooding_heartbeat': np.int64(120), 'cyberattack_ocpp16_fdi_chargingprofile': np.int64(45), 'cyberattack_ocpp16_unauthorized_access': np.int64(45), 'normal': np.int64(60)}
Episode 3 (shift=0.20): 300 samples — {'cyberattack_ocpp16_doc_idtag': np.int64(30), 'cyberattack_ocpp16_dos_flooding_heartbeat': np.int64(60), 'cyberattack_ocpp16_fdi_chargingprofile': np.int64(120), 'cyberattack_ocpp16_unauthorized_access': np.int64(45), 'normal': np.int64(45)}
Episode 4 (shift=0.08): 300 samples — {'cyberattack_ocpp16_doc_idtag': np.int64(60), 'cyberattack_ocpp16_dos_flooding_heartbeat': np.int64(75), 'cyberattack_o